<a id="location-detection"></a>
# VideoDB Understanding: Location Detection

Describe the visible environment without guessing unsupported cities or venues.


<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/location-detection/location-detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install, connect, and choose a video

In [ ]:
!pip install -q videodb python-dotenv pandas

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()
if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
collection = conn.get_collection()
print("Connected to VideoDB")

In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"
video = collection.upload(VIDEO_URL)

# To use an existing video instead:
# video = collection.get_video("m-...")

print("Video:", video.id)
video.play()

<a id="configuration"></a>
## 2. Configure location detection

The analyzer favors observable descriptions such as indoor office, outdoor street, lighting, and broad environment.

This analyzer uses sampled frames and returns one result for every segment.

In [ ]:
ANALYZER = {
    "type": "location_detection",
    "name": "location",
    "sampling": {"strategy": "uniform", "frame_count": 3},
    "config": {"model": "basic"},
}

print("Analyzer configuration:")
ANALYZER

## 3. Run the analyzer

In [ ]:
understanding = video.understand(
    analyzers=[ANALYZER],
    segmentation={"type": "shot", "threshold": 30},
)

understanding.wait_until_complete(timeout=3600, poll_interval=15)
analyzer = understanding.get_analyzer("location", refresh=True)

print("Understanding complete")
print(f"ID: {understanding.id}")
print(f"Analyzer: {analyzer.name} ({analyzer.type})")
print(f"Status: {analyzer.status}")

<a id="output"></a>
## 4. Inspect the output

Each scene contains `data.location`, grounded in visible setting attributes.

In [ ]:
import pandas as pd

output = analyzer.get_output()
rows = [
    {"start": scene.get("start"), "end": scene.get("end"), **(scene.get("data") or {})}
    for scene in output.get("scenes", [])
]

print(f"Scenes: {len(rows)}")
pd.DataFrame(rows).reindex(columns=['start', 'end', 'location'])

## 5. Show results beside VideoDB frames

The midpoint frame provides visual evidence for each timestamped result.

In [ ]:
import html
import json

from IPython.display import HTML, display


def thumbnail_url(timestamp):
    asset = video.generate_thumbnail(time=max(float(timestamp), 0.001))
    return asset.url or asset.generate_url()


cards = []
for scene in output.get("scenes", [])[:4]:
    start = float(scene.get("start") or 0)
    end = float(scene.get("end") or start)
    value = (scene.get("data") or {}).get("location")
    cards.append(f"""
    <div style="width:280px;border:1px solid #ddd;border-radius:10px;overflow:hidden">
      <img src="{thumbnail_url((start + end) / 2)}" style="width:100%;display:block"/>
      <div style="padding:10px"><b>{start:.1f}s–{end:.1f}s</b>
      <pre style="white-space:pre-wrap">{html.escape(json.dumps(value, ensure_ascii=False, indent=2))}</pre></div>
    </div>""")

display(HTML('<div style="display:flex;gap:12px;flex-wrap:wrap">' + ''.join(cards) + '</div>'))

## 6. Use the output

The most useful index fields are `location`. Continue with the [Indexing guide](../../indexing/indexing_guide.ipynb), or feed this analyzer into a [multi-analyzer pipeline](../multi-analyzer-pipelines.ipynb).

## Optional cleanup

In [ ]:
DELETE_RUN = False
if DELETE_RUN:
    understanding.delete()